# Today is about putting the pipeline ine a container

Yesterday, my pipeline ran from a terminal on my computer. Today it goes into a Docker container, with Redis beside it. The container has no notebook state, no secrets baked in, and keeps its data across restarts. `day19_tools.zip` is attached.


# Theory
**1. Image and container.** the image is the frozen recipe: Python, my packages and my code. A container is one running copy of it. Deleting the container and the image stays. The Dockerfile installs packages before copying your code, so a code change doesn't reinstall everything.

**2. "localhost" means "myself"**. Inside a container, `localhost` is the container, my pipeline's `Redis(host="localhost")` would fail here. Compose starts Redis as a second container named `redis`, and the patch makes the host come from an environment variable. The default stays `localhost`, so my notebook and terminal behave as before.

**3. Secrets are never baked in.** An Image keeps everything copied into it, so anyone with the image gets the secrets. My keys stay in `.env` and reach the container only when it starts. `.dockerignore` is an allow-list: only the files it names go into the image.

**4. Volumes keep data.** Deleting a container deletes what it wrote. Three names volumes keep what matters: the SQLite checkpoints, The Redis data, and the downloaded models. A named volume is used instead of a Windows folder because SQLite's file locking breaks the Windows file boundary.

**5. Least priviledge.** The app  runs as a normal user, not root. `cap_drop:ALL` removes the container's special Linux powers. Only my computer can reach port 8080.

**6. Two health checks.** `/healthz` asks whether the process is alive, and Docker uses it. `/readyz` asks whether Redis, Mongo and Neo4j answer, That one is for a load balanver. If Docker used `/readyz`, a brief Atlas hiccup would make it restart a perfectly good container.

**7. A clean stop.** `docker stop` sends a polite shutdown signal (SIGTERM). The server finishes the question it is answering, then exits. The grace period is 150 s, longer than the 120 s request timeout. Exit code 143 means "ended by SIGTERM" and is normal.

**8. CPU PyTorch.** A container has no GPU. The GPU build of PyTorch is about 2.5 GB bigger and useless there. The requirements builder drops Windows-only and GPU-only packages for you.

In [2]:
# preflight

import os, sys, subprocess, time, json, pathlib, re, threading

# --- 1. The Day 19 files and the Day 18 files must be in this folder
FILES = ["container_check.py", "container_patch.py", "make_requirements.py", "make_container_files.py",
         "docker_healthcheck.py", "test_container.py",
         "freeze_kernel.py", "serve.py", "api_layer.py"]            # the last three came from Day 17/18
absent = [f for f in FILES if not os.path.exists(f)]
assert not absent, f"❌ Missing from {os.getcwd()}: {absent}  (extract day19_tools.zip into this folder)"
for must in ("rag_pipeline/pipeline.py", "rag_pipeline/_parity_expected.json", ".env"):
    assert os.path.exists(must), f"❌ {must} is missing. It comes from Day 18 (freeze) / Day 18 Cell 7 (.env)."

# --- 2. Packages the tools need
for mod, pkg in (("yaml", "pyyaml"), ("packaging", "packaging"), ("dotenv", "python-dotenv"), ("httpx", "httpx")):
    try:
        __import__(mod)
    except ImportError:
        raise AssertionError(f"❌ {pkg} is not installed. Run:  %pip install {pkg}")

# --- 3. A helper that runs docker commands and gives back (exit code, text)
def dc(*args, timeout=1800):
    try:
        r = subprocess.run(["docker", *args], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=timeout)
    except FileNotFoundError:
        return 127, "docker was not found on this computer"
    return r.returncode, (r.stdout + r.stderr).strip()

# --- 4. Is Docker running? (Cells 1-6 work without it; the terminal step does not)
code, out = dc("info", "--format", "{{.ServerVersion}} {{.MemTotal}}")
if code == 0:
    version, mem = out.split()[:2]
    gb = int(mem) / 1024**3
    print(f"✅ Docker is running (engine {version}, {gb:.1f} GB of memory available to it)")
    if gb < 6:
        print("⚠️  Docker has less than 6 GB. The embedding model needs about 3-4 GB while it runs. "
              "In Docker Desktop: Settings > Resources, or a .wslconfig file with  memory=8GB")
else:
    print("⚠️  Docker is not running or not installed:", out[:200])
    print("   Cells 1-6 still work. Start Docker Desktop (wait for the whale icon to stop moving) before the terminal step.")

# --- 5. The Day 18 concurrency patch must be in place if .env asks for more than one slot
env_text = pathlib.Path(".env").read_text(encoding="utf-8-sig")
m = re.search(r"(?m)^RAG_MAX_CONCURRENT=(\d+)", env_text)
slots = int(m.group(1)) if m else 1
if slots > 1 and "UsageProxy" not in pathlib.Path("rag_pipeline/pipeline.py").read_text(encoding="utf-8"):
    raise AssertionError(f"❌ .env says RAG_MAX_CONCURRENT={slots} but the Day 18 patch is not in pipeline.py. Redo Day 18 Cell 5.")
print(f"✅ ready. Python {sys.version.split()[0]} | folder: {os.getcwd()} | slots in .env: {slots}")


✅ Docker is running (engine 29.8.2, 3.8 GB of memory available to it)
⚠️  Docker has less than 6 GB. The embedding model needs about 3-4 GB while it runs. In Docker Desktop: Settings > Resources, or a .wslconfig file with  memory=8GB
✅ ready. Python 3.11.14 | folder: c:\Users\rodne\Desktop\4 MONTHS JOURNEY\MONTH4-2 | slots in .env: 2


In [3]:
# The tools will test themselves in about 20 seconds

# The tools test themselves before touching your files. About 20 seconds. No Docker needed.
t0 = time.time()
r = subprocess.run([sys.executable, "test_container.py"], capture_output=True, text=True, encoding="utf-8", errors="replace")
lines = r.stdout.splitlines()
print(next((l for l in lines if l.endswith("passed")), "NO RESULT"), f"({time.time() - t0:.0f}s)")
for l in lines:
    if l.startswith("FAIL"):
        print("   ", l)
if r.returncode != 0:
    print(r.stdout[-2500:], r.stderr[-1500:])
assert r.returncode == 0, "❌ a tool test failed: read the output above before touching your files"
print("✅ all tool tests pass")

38/38 passed (17s)
✅ all tool tests pass


In [6]:
# Checking my folder, it Changes nothing

# Reads your frozen package, your .env (names only) and this folder. Changes nothing except a hidden byte in .env, if there is one.
sys.path.insert(0, os.getcwd())
from container_check import check_container_readiness, fix_env_file

for line in fix_env_file(".env"):
    print("🔧", line)

check = check_container_readiness("rag_pipeline", ".")
print(check.summary())

What the Docker files will handle:
   - The Redis host/port are written into the code (localhost). The next cell makes them come from REDIS_HOST / REDIS_PORT (same defaults as today), and compose adds a Redis container.
   - 'checkpoints/rag_checkpoints.db' is state the app writes: it goes in a Docker volume so it survives restarts (a bind mount from Windows breaks SQLite file locking).
   - 'day14_visual_index.json' exists here and the code opens it: it will be copied into the image.
   - '' exists here and the code opens it: it will be copied into the image.
Good to know:
   - Models downloaded on the first start (kept in the 'models' volume afterwards): BAAI/bge-m3. The first start can take several minutes and a few GB.

✅ nothing blocks the container


In [7]:
# making Redis' address configurable

# Inside a container, "localhost" is the container itself. Make Redis' address come from REDIS_HOST / REDIS_PORT,
# with the SAME defaults as today, so this notebook and your terminal keep working exactly as before.
from container_patch import make_redis_env_driven, ContainerPatchError
from freeze_kernel import verify_in_fresh_process

if check.redis_mode == "external":
    print("ℹ️  Redis is on a remote server: nothing to change.")
else:
    try:
        print(make_redis_env_driven("rag_pipeline/pipeline.py"))
    except ContainerPatchError as e:
        raise AssertionError("❌ " + str(e))

# Same proof as Day 18: load the package in a brand-new process and compare it with this kernel.
t0 = time.time()
problems = verify_in_fresh_process("rag_pipeline")
print(f"(fresh process took {time.time() - t0:.0f}s)")
print("\n".join("❌ " + p for p in problems) if problems else "✅ the package behaves exactly as before")

patched 1 place(s): Redis host/port/url now come from REDIS_HOST / REDIS_PORT / REDIS_URL (same defaults as before)
(fresh process took 71s)
✅ the package behaves exactly as before


In [9]:
# The pachake list for the image

# The list of packages the image installs: what your code imports + everything they need, at the versions you tested with.
from make_requirements import write_requirements

reqs = write_requirements("rag_pipeline", ".")
print(reqs.summary())
print("\nPackages your code imports directly:", ", ".join(reqs.top_level))

Python 3.11 | 19 packages found from your imports, 113 pinned in total (everything they need, exact versions)
PyTorch will be installed in its CPU build: torch==2.10.0
1 Linux-only helper packages are not installed on this machine (pip fetches them inside the image): cuda-bindings
wrote: requirements.docker.txt, requirements.torch.txt

Packages your code imports directly: fastapi, groq, langchain-community, langgraph, langgraph-checkpoint-sqlite, llama-index-core, llama-index-embeddings-huggingface, llama-index-storage-docstore-mongodb, llama-index-vector-stores-mongodb, neo4j, numpy, pydantic, pymongo, python-dotenv, redis, requests, sentence-transformers, starlette, uvicorn


In [10]:
# Writing the Docker files

# Writes Dockerfile, docker-compose.yml, .dockerignore and docker_healthcheck.py (old ones are kept as *.previous).
from make_container_files import make_container_files

written = make_container_files("rag_pipeline", ".", check, reqs)
print("wrote:", ", ".join(written))

code, out = dc("compose", "config", "-q")
print("docker-compose.yml:", "✅ valid" if code == 0 else "❌ " + out[-800:])

gi = pathlib.Path(".gitignore")
lines = gi.read_text(encoding="utf-8").splitlines() if gi.exists() else []
add = [x for x in ("*.before_container_patch", ".env.before_bom_fix") if x not in lines]
if add:
    gi.write_text("\n".join(lines + add) + "\n", encoding="utf-8")

wrote: Dockerfile, .dockerignore, docker-compose.yml, docker_healthcheck.py
docker-compose.yml: ✅ valid


In [17]:
# Smoke test

# Run AFTER the terminal step (docker compose up -d).
import httpx
from dotenv import dotenv_values
KEY = dotenv_values(".env")["RAG_API_KEYS"].split(",")[0].split(":")[0]
BASE, H = "http://127.0.0.1:8080", {"X-API-Key": KEY}

def wait_ready(max_seconds=900):
    """Poll /readyz until it says 200. The very first start downloads the embedding model, so it can take minutes."""
    t0, last = time.time(), "nothing yet"
    while time.time() - t0 < max_seconds:
        try:
            r = httpx.get(f"{BASE}/readyz", timeout=5)
            if r.status_code == 200:
                return time.time() - t0
            last = f"readyz {r.status_code} {r.text[:200]}"
        except Exception as e:
            last = type(e).__name__
        time.sleep(3)
    raise AssertionError(f"❌ not ready after {max_seconds}s (last: {last}). Read the server's log:  docker compose logs --tail 80 rag")

print(f"ready after {wait_ready():.0f}s")
print("healthz :", httpx.get(f"{BASE}/healthz").json())
r = httpx.get(f"{BASE}/readyz")
print("readyz  :", r.status_code, r.json())
assert r.status_code == 200, "❌ a dependency is down: read the log above and  docker compose logs --tail 80 rag"
print("no key  :", httpx.post(f"{BASE}/ask", json={"question": "hi"}).status_code, "(want 401)")

t0 = time.time()
r = httpx.post(f"{BASE}/ask", json={"question": "What was Apple's total net sales for fiscal year 2024?"}, headers=H, timeout=300)
print(f"/ask    : {r.status_code} in {time.time() - t0:.1f}s")
print(json.dumps(r.json(), indent=2)[:900])
assert r.status_code == 200 and "391" in r.json()["answer"], "❌ the container did not answer correctly"
print("\n✅ the pipeline answers from inside a container")

ready after 2s
healthz : {'status': 'ok'}
readyz  : 200 {'status': 'ready', 'checks': {'redis': 'ok', 'mongo': 'ok', 'neo4j': 'ok'}}
no key  : 401 (want 401)
/ask    : 200 in 31.3s
{
  "request_id": "c1f8ef1c271747f286793b0a76012b08",
  "session_id": "3715459998b2414f8c324f66044b2cdf",
  "answer": "Apple\u2019s total net sales for fiscal year\u202f2024 were **$391,035\u202fmillion** (approximately $391.0\u202fbillion).",
  "classification": "SIMPLE",
  "retrieval_source": "local",
  "reviewer_verdict": "PASS",
  "warning": null,
  "langsmith_run_id": null,
  "elapsed_ms": 30882
}

✅ the pipeline answers from inside a container


In [18]:
# Does my data survive? (one real question)

# Does your data survive a restart, and a full tear-down and start-up? (one real question)
def snapshot():
    c1, keys = dc("compose", "exec", "-T", "redis", "redis-cli", "--scan", "--pattern", "conversation:*")
    c2, files = dc("compose", "exec", "-T", "rag", "python", "-c",
                   "import os, json; d='/app/checkpoints'; "
                   "print(json.dumps({f: os.path.getsize(os.path.join(d, f)) for f in os.listdir(d) if not f.endswith(('-wal', '-shm'))}))")
    assert c1 == 0 and c2 == 0, f"❌ could not look inside the containers:\n{keys}\n{files}"
    return set(keys.split()), json.loads(files.splitlines()[-1])

r = httpx.post(f"{BASE}/ask", json={"question": "What was Apple's net income in fiscal 2024?", "session_id": "day19-persist"}, headers=H, timeout=300)
assert r.status_code == 200, r.text[:300]
keys0, files0 = snapshot()
print("conversation keys in Redis:", sorted(keys0))
print("files in checkpoints/     :", files0)
assert keys0, "❌ Redis holds no conversation after a question: the pipeline did not save memory (a pipeline matter, not a Docker one)"
assert files0 and all(size > 0 for size in files0.values()), "❌ no checkpoint file was written inside the container"

code, out = dc("compose", "restart")
assert code == 0, out
print(f"restart            -> ready again after {wait_ready():.0f}s")
keys1, files1 = snapshot()
assert keys0 <= keys1 and set(files0) <= set(files1), f"❌ lost after restart: {keys0 - keys1} {set(files0) - set(files1)}"

dc("compose", "down")                      # containers are deleted; the volumes (your data) are NOT
code, out = dc("compose", "up", "-d")
assert code == 0, out
print(f"down, then up -d   -> ready again after {wait_ready():.0f}s (the models were not downloaded again)")
keys2, files2 = snapshot()
assert keys0 <= keys2 and set(files0) <= set(files2), f"❌ lost after down/up: {keys0 - keys2} {set(files0) - set(files2)}"
print("\n✅ conversation memory (Redis) and checkpoints (SQLite) survive restarts and re-creation")

conversation keys in Redis: ['conversation:rodney-api:3715459998b2414f8c324f66044b2cdf', 'conversation:rodney-api:day19-persist']
files in checkpoints/     : {'rag_checkpoints.db': 4096}
restart            -> ready again after 401s
down, then up -d   -> ready again after 447s (the models were not downloaded again)

✅ conversation memory (Redis) and checkpoints (SQLite) survive restarts and re-creation


In [19]:
# What's inside the image?


# What is INSIDE the image? Nothing secret, nothing of yours that it should not hold, and not running as root.
from container_check import secret_values

code, out = dc("run", "--rm", "--entrypoint", "sh", "rag-api:local", "-c", "id -u; ls -A /app")
assert code == 0, out
uid, *files = out.split()
allowed = {"rag_pipeline", "serve.py", "api_layer.py", "docker_healthcheck.py", "checkpoints", "requirements.docker.txt", "requirements.torch.txt"} | {p.split("/")[0] for p in check.copy_paths}
unexpected = sorted(set(files) - allowed)
print("runs as user id:", uid, "(want 10001, not 0)")
print("files in /app  :", sorted(files))
assert uid == "10001", "❌ the container runs as root"
assert not unexpected, f"❌ these should not be in the image: {unexpected}"

# Look for every secret value from .env inside the image. The values travel through stdin, never on a command line.
values = secret_values(".env")
scan = ("import sys, os, json\n"
        "vals = [l.rstrip('\\n') for l in sys.stdin if l.strip()]\n"
        "hits = set()\n"
        "for root, _, names in os.walk('/app'):\n"
        "    for n in names:\n"
        "        p = os.path.join(root, n)\n"
        "        try:\n"
        "            data = open(p, 'rb').read()\n"
        "        except Exception:\n"
        "            continue\n"
        "        if any(v.encode() in data for v in vals):\n"
        "            hits.add(p)\n"
        "print(json.dumps(sorted(hits)))\n")
r = subprocess.run(["docker", "run", "--rm", "-i", "--entrypoint", "python", "rag-api:local", "-c", scan],
                   input="\n".join(values), capture_output=True, text=True, encoding="utf-8", errors="replace")
assert r.returncode == 0, r.stderr[-500:]
hits = json.loads(r.stdout.splitlines()[-1])
assert not hits, f"❌ secret values found inside the image files: {hits}"

meta = dc("image", "inspect", "rag-api:local")[1] + dc("history", "--no-trunc", "rag-api:local")[1]
leaked = [i for i, v in enumerate(values) if v in meta]
assert not leaked, "❌ a secret value is baked into the image's settings or build history"
size = int(dc("image", "inspect", "--format", "{{.Size}}", "rag-api:local")[1])
print(f"checked {len(values)} secret value(s) from .env: none is inside the image")
print(f"image size: {size / 1024**3:.2f} GB")
print("\n✅ the image holds code and packages only")

runs as user id: 10001 (want 10001, not 0)
files in /app  : ['api_layer.py', 'checkpoints', 'day14_visual_index.json', 'docker_healthcheck.py', 'rag_pipeline', 'requirements.docker.txt', 'requirements.torch.txt', 'serve.py']
checked 16 secret value(s) from .env: none is inside the image
image size: 2.08 GB

✅ the image holds code and packages only


In [20]:
# a Clean stop, (one real question)

# What happens to a question that is being answered when the container is told to stop? (one real question)
box = {}
def ask():
    t0 = time.time()
    r = httpx.post(f"{BASE}/ask", json={"question": "What was Apple's net income in fiscal 2024, and how did it change from fiscal 2023?",
                                         "session_id": "day19-stop"}, headers=H, timeout=300)
    box["status"], box["secs"] = r.status_code, time.time() - t0
th = threading.Thread(target=ask)
th.start()
time.sleep(2)                                           # the question is now being worked on
t0 = time.time()
code, out = dc("compose", "stop", "rag")
stop_secs = time.time() - t0
th.join(300)
cid = dc("compose", "ps", "-a", "-q", "rag")[1].split()[0]
exit_code = dc("inspect", "--format", "{{.State.ExitCode}}", cid)[1]
print(f"the question finished with status {box.get('status')} after {box.get('secs', 0):.1f}s; the stop took {stop_secs:.1f}s; exit code {exit_code}")
if box.get("secs", 0) < 4:
    print("ℹ️  the question was too quick to prove much: run this cell again if you want a stronger test")
assert box.get("status") == 200, "❌ the question in progress was cut off by the stop"
assert exit_code in ("0", "143"), f"❌ unexpected exit code {exit_code} (0 and 143 are the normal ones)"
dc("compose", "start", "rag")
print(f"started again, ready after {wait_ready():.0f}s")
print("\n✅ a stop waits for the question in progress (up to 150 s), then shuts down cleanly")

the question finished with status 200 after 61.4s; the stop took 87.0s; exit code 143
started again, ready after 465s

✅ a stop waits for the question in progress (up to 150 s), then shuts down cleanly


In [21]:
# Optional, when i finish for the day

# Optional, when you finish for the day: stop everything. Your data is kept.
code, out = dc("compose", "down")
print(out or "stopped")
print("Kept: the volumes models, checkpoints and redis_data. Start again any time with:   docker compose up -d")

Container rag-api-rag-1 Stopping 
 Container rag-api-rag-1 Stopped 
 Container rag-api-rag-1 Removing 
 Container rag-api-rag-1 Removed 
 Container rag-api-redis-1 Stopping 
 Container rag-api-redis-1 Stopped 
 Container rag-api-redis-1 Removing 
 Container rag-api-redis-1 Removed 
 Network rag-api_default Removing 
 Network rag-api_default Removed
Kept: the volumes models, checkpoints and redis_data. Start again any time with:   docker compose up -d
